In [ ]:
import cv2
import os
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# Primeiro frame

In [ ]:
# Carrega primeiro frame  # lembrar de carregar certo em tons de cinza
first_frame = cv2.imread(image_paths[0])
print(f"Primeiro frame carregado: {first_frame.shape}")

first_gt = all_ground_truths[0]
print(f"Ground Truth do primeiro frame: {first_gt}")

# Mostra o frame com a bbox
first_frame_with_bbox = first_frame.copy()
x, y, w, h = map(int, first_gt)
cv2.rectangle(first_frame_with_bbox, (x, y), (x + w, y + h), (0, 255, 0), 2) 

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(first_frame_with_bbox, cv2.COLOR_BGR2RGB))
plt.title('Primeiro Frame - Objeto a Rastrear')
plt.axis('off')
plt.show()


In [ ]:
# --- PASSO 2: EXTRAIR E PRÉ-PROCESSAR PATCH ---
print("\n🔍 PASSO 2: Extraindo e pré-processando patch...")

# Extrai patch do objeto
patch = tracker_utils.extract_patch(first_frame, first_gt)

# Mostra o patch ORIGINAL (antes do pré-processamento)
plt.figure(figsize=(8, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
plt.title(f'Patch Original\n shape:{patch.shape}')
plt.axis('off')


In [ ]:
import cv2
import numpy as np

def preprocess_frame(frame, n_levels=20, adaptive=False):
    """
    Pré-processa um frame para entrada em modelos baseados em bits.
    """

    # --------------------------------------------------
    # Garantias de compatibilidade
    # --------------------------------------------------
    if frame.ndim == 3:
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        frame_gray = frame

    # força mesmo tamanho do background
    if frame_gray.shape != background_model.shape:
        frame_gray = cv2.resize(
            frame_gray,
            (background_model.shape[1], background_model.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    frame_gray = frame_gray.astype(np.uint8)

    # --------------------------------------------------
    # 1. Remoção do fundo
    # --------------------------------------------------
    diff = cv2.absdiff(frame_gray, background_model)

    # --------------------------------------------------
    # 2. Binarização
    # --------------------------------------------------
    if adaptive:
        binary = cv2.adaptiveThreshold(
            diff, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            11, 2
        )
    else:
        _, binary = cv2.threshold(
            diff, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

    # --------------------------------------------------
    # 3. Codificação termômetro
    # --------------------------------------------------
    binary_norm = binary.astype(np.float32) / 255.0

    thresholds = np.linspace(0, 1, n_levels, endpoint=False)
    thermo = (binary_norm[..., None] > thresholds).astype(np.uint8)

    return thermo.flatten()


In [ ]:
# Pré-processa o patch
first_pattern = preprocess_frame(patch)
print(f"Primeiro Pattern pré-processado: {first_pattern.shape}")

In [ ]:
# --- PASSO 3: INSTANCIANDO CLUSWISARD ---

# Parâmetros simples
parameters = {
    "INPUT_PATTERN_SIDE": 32,                  
    "CLUSWISARD_ADDRESS_SIZE": 2,          
    "ACCEPTABLE_ACTIVATION_SCORE": 0.3,       
    "MIN_ACCEPTABLE_ACTIVATION_SCORE": 0.1,    
    "CLUSWISARD_MIN_SCORE": 0.3,               
    "CLUSWISARD_THRESHOLD": 20,                 
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 20,      
    "CLUSWISARD_BLEACHING_ACTIVATED": False,     
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "INITIAL_FRAME": 1,
    "LAST_FRAME": 25,
    "MAX_SEARCH_WINDOW_SCALE": 2,
}

# Instancia ClusWisard

clus = ClusWisard(
            parameters['CLUSWISARD_ADDRESS_SIZE'], # address size
            parameters['CLUSWISARD_MIN_SCORE'], # min score
            parameters['CLUSWISARD_THRESHOLD'], # threshold
            parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'], # discriminator limit
            bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
            returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
            returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
            returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
        )

In [ ]:
# Treina com o patch do objeto
clus.train([first_pattern.tolist()], ["object"])
print("✅ ClusWisard treinado com o patch do primeiro frame!")

In [ ]:
# --- PASSO 4: TESTAR SE RECONHECE O PRÓPRIO PATCH ---
print("\n✅ PASSO 4: Testando autorreconhecimento...")

# Classifica o MESMO patch usado no treinamento
result = clus.classify([first_pattern.tolist()])[0]
activation = result.get("activationDegree", -1)

print(f"Ativação com o próprio patch de treinamento: {activation}")
print(result)

# Região de Busca

In [ ]:
import numpy as np  # importa o NumPy para operações numéricas e trigonométricas

def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1.5, 
    step_size=1,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        
        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


### Gerando regiões para o primeiro frame

In [ ]:
# Parâmetros
search_window_scale = 2.0
step_size = 1          # quanto menor -> mais granular

search_gen = generate_search_regions_circular(
    prev_bbox = (16.0, 30.0, 34.0, 39.0),
    frame_shape = first_frame.shape,
    search_region_scale = search_window_scale,
    step_size = step_size
)

### Classificação das regiões de busca do primeiro frame

In [ ]:
# --- PASSO 7: CLASSIFICAR CADA SEARCH REGION ---
def classify_search_regions(clus, regions, frame):
    results = []
    for i, region in enumerate(regions):
        # Extrai o patch
        patch_region = tracker_utils.extract_patch(frame, region)
        
        if patch_region.size == 0:
            print(f"❌ Region {i}: Patch vazio - ignorando")
            continue
            
        # Pré-processa
        pattern_region = preprocess_frame(patch_region)
        
        # Classifica com ClusWisard
        result = clus.classify([pattern_region.tolist()])[0]
        activation = result.get("activationDegree", -1)
        
        results.append({
            'region_id': i,
            'bbox': region,
            'activation': activation
        })
        
        print(f"Region {i}: ativação = {activation:.4f}")

    # Ordena por ativação (maior primeiro)
    results_sorted = sorted(results, key=lambda x: x['activation'], reverse=True)

    print("\n🏆 RANKING DAS SEARCH REGIONS:")
    for i, result in enumerate(results_sorted):
        print(f"{i+1}º - Region {result['region_id']}: ativação = {result['activation']:.4f}")

    # Mostra a melhor região
    best_region = results_sorted[0]
    print(f"\n🎯 MELHOR REGIÃO: Region {best_region['region_id']} com ativação {best_region['activation']:.4f}")
    return results_sorted

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_top_regions_grid(frame, results_sorted, top_n=5, cols=3):
    """
    Mostra as N regiões mais bem classificadas em um grid.

    Parâmetros:
        frame: imagem original (BGR)
        results_sorted: lista retornada de classify_search_regions()
        top_n: número de regiões a exibir
        cols: número de colunas no grid
    """
    top_n = min(top_n, len(results_sorted))
    rows = (top_n + cols - 1) // cols

    plt.figure(figsize=(4 * cols, 4 * rows))

    h_frame, w_frame = frame.shape[:2]

    for i in range(top_n):
        region_info = results_sorted[i]
        region_id = region_info["region_id"]
        activation = region_info["activation"]

        # --- ✅ Converte coordenadas para inteiros
        x, y, w, h = [int(round(v)) for v in region_info["bbox"]]

        # --- ⚠️ Garante que os índices estão dentro dos limites do frame
        x = max(0, min(x, w_frame - 1))
        y = max(0, min(y, h_frame - 1))
        w = max(1, min(w, w_frame - x))
        h = max(1, min(h, h_frame - y))

        # --- Extrai o patch da região
        patch = frame[y:y+h, x:x+w]
        if patch.size == 0:
            print(f"⚠️ Region {region_id} vazia - ignorando exibição")
            continue

        patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)

        # --- Mostra patch no grid
        plt.subplot(rows, cols, i + 1)
        plt.imshow(patch_rgb)
        plt.title(f"#{i+1} Reg {region_id}\nAct={activation:.3f}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
results_sorted = classify_search_regions(clus, search_gen, first_frame)


In [ ]:
show_top_regions_grid(first_frame, results_sorted, top_n=1000, cols=20)
#analisar pro segundo frame para avaliar a velocidade (definir step e região limite)

### Vídeo exemplo de região de busca para o primeiro frame

In [ ]:
import cv2
import numpy as np

max_regions = 300
output_path = "search_circular.mp4"
fps = 1
accumulate = True

# Garante formato correto
if first_frame.ndim == 2:
    first_frame = cv2.cvtColor(first_frame, cv2.COLOR_GRAY2BGR)
elif first_frame.dtype != np.uint8:
    first_frame = (first_frame * 255).astype(np.uint8)

h, w_img = first_frame.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(output_path, fourcc, fps, (w_img, h))

if not video_writer.isOpened():
    raise RuntimeError("❌ Falha ao abrir o VideoWriter. Verifique codec ou caminho.")

drawn_regions = []
for i, region in enumerate(search_gen):
    if i >= max_regions:
        break

    frame_copy = first_frame.copy()

    if accumulate:
        for (px, py, pw, ph) in drawn_regions:
            cv2.rectangle(frame_copy, (int(px), int(py)), (int(px+pw), int(py+ph)), (0, 0, 255), 1)

    x_r, y_r, w_r, h_r = map(int, region)
    cv2.rectangle(frame_copy, (x_r, y_r), (x_r + w_r, y_r + h_r), (0, 255, 0), 2)
    cv2.putText(frame_copy, f"Region {i}", (x_r, max(0, y_r - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Converte para BGR antes de gravar
    frame_copy_bgr = cv2.cvtColor(frame_copy, cv2.COLOR_RGB2BGR)
    video_writer.write(frame_copy_bgr)

    drawn_regions.append((x_r, y_r, w_r, h_r))

video_writer.release()
print(f"✅ Vídeo salvo em: {output_path}  (frames: {len(drawn_regions)})")


In [ ]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# Carrega imagens
# --------------------------------------------------
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# --------------------------------------------------
# Carrega ground truths
# --------------------------------------------------
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# --------------------------------------------------
# Frame 0 e GT
# --------------------------------------------------
frame0 = cv2.imread(image_paths[0], cv2.IMREAD_GRAYSCALE)
assert frame0 is not None

x, y, w, h = map(int, all_ground_truths[0])
H, W = frame0.shape

# --------------------------------------------------
# Máscara de fundo
# --------------------------------------------------
background_mask = np.ones((H, W), dtype=np.uint8)
background_mask[y:y+h, x:x+w] = 0

# --------------------------------------------------
# Modelo de fundo (média)
# --------------------------------------------------
background_model = frame0.astype(np.float32)
background_model[background_mask == 0] = 0
background_model /= np.maximum(background_mask, 1)
background_model = background_model.astype(np.uint8)

In [ ]:
import cv2
import numpy as np

max_regions = 300
output_path = "search_circular.mp4"
fps = 10
accumulate = True

h, w_img = first_frame.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(output_path, fourcc, fps, (w_img, h))
print("Aberto?", video_writer.isOpened())

drawn_regions = []

for i, region in enumerate(search_gen):
    if i >= max_regions:
        break

    frame_copy = first_frame.copy()

    # Garante que todas as dimensões estão iguais
    if frame_copy.shape[:2] != (h, w_img):
        frame_copy = cv2.resize(frame_copy, (w_img, h))

    if accumulate:
        for (px, py, pw, ph) in drawn_regions:
            cv2.rectangle(frame_copy, (int(px), int(py)), (int(px+pw), int(py+ph)), (0, 0, 255), 1)

    x_r, y_r, w_r, h_r = map(int, region)
    cv2.rectangle(frame_copy, (x_r, y_r), (x_r + w_r, y_r + h_r), (0, 255, 0), 2)
    cv2.putText(frame_copy, f"Region {i}", (x_r, max(0, y_r - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    video_writer.write(cv2.cvtColor(frame_copy, cv2.COLOR_RGB2BGR))
    drawn_regions.append((x_r, y_r, w_r, h_r))

video_writer.release()
print(f"✅ Vídeo salvo em: {output_path} (frames: {len(drawn_regions)})")


# Tracker

In [ ]:
import json

In [ ]:
# --- LOOP DE TRACKING QUADRO A QUADRO ---
print("\n🔄 Iniciando tracking sobre todos os frames...")

prev_bbox = first_gt  
all_predictions = [prev_bbox]  # guarda todas as previsões

for frame_idx, frame_path in enumerate(image_paths[1:100], start=1):
    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} ({frame_path}) não carregado!")
        continue

    print(f"\n📷 Frame {frame_idx}: {frame_path}")
    
    # --- PASSO 1: Gerar regiões de busca ---
    search_gen = generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=frame.shape,
        search_region_scale=2,
        step_size=1
    )
    
    all_regions = list(search_gen)
    print(f"🔍 Geradas {len(all_regions)} regiões para o frame {frame_idx}")
    
    # --- PASSO 2: Classificar regiões ---
    results = []
    for i, region in enumerate(all_regions):
        patch_region = tracker_utils.extract_patch(frame, region)
        if patch_region.size == 0:
            continue
        
        pattern_region = preprocess_frame(patch_region) 
        
        result = clus.classify([pattern_region.tolist()])[0]
        activation = result.get("activationDegree", -1)
        
        results.append({
            'region_id': i,
            'bbox': region,
            'activation': activation,
            'patch': pattern_region  # armazenamos para possível retraining
        })
        
    if not results:
        print("⚠️ Nenhuma região válida encontrada, mantendo prev_bbox.")
        all_predictions.append(prev_bbox)
        continue
    
    # --- PASSO 3: Escolher melhor região ---
    best_region = max(results, key=lambda r: r['activation'])
    prev_bbox = best_region['bbox']
    all_predictions.append(prev_bbox)
    
    print(f"🎯 Melhor região frame {frame_idx}: {best_region['bbox']} ativação={best_region['activation']:.4f}")
    
    # --- Visualizar patch da melhor região ---
    best_patch = tracker_utils.extract_patch(frame, best_region['bbox'])
    
    # # --- Retreinar ClusWisard com o patch atual ---
    # if best_region['activation'] < 0.7:
    #     print(f"⚡ Ativação baixa ({best_region['activation']:.4f}) → Retreinando ClusWisard com o patch atual...")
    #     clus.train([best_region['patch'].tolist()], ["object"])
    #     json_str = clus.json()

        # transforma em dict
        # clus_dict = json.loads(json_str)

        # clusters = clus_dict["clusters"]

        # # Número de clusters (classes)
        # num_clusters = len(clusters)

        # # Número de discriminadores por cluster
        # discriminadores_por_cluster = {nome: len(c["discriminators"]) for nome, c in clusters.items()}

        # print("Número de clusters:", num_clusters)
        # print("Discriminadores por cluster:", discriminadores_por_cluster)

    plt.figure()
    plt.title(f"Frame {frame_idx} - Melhor Região")
    plt.imshow(best_patch)

print("\n✅ Tracking concluído!")


In [ ]:
import cv2
import os
import shutil


# --- Pasta de saída ---
output_dir = "resultados/"

# Apaga a pasta se existir
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Cria nova pasta vazia
os.makedirs(output_dir, exist_ok=True)
# --- Salvar cada frame com a bbox prevista ---
for frame_idx, frame_path in enumerate(image_paths[1:]):
    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} não carregado!")
        continue

    # Pega a previsão correspondente
    if frame_idx < len(all_predictions):
        x, y, bw, bh = map(int, all_predictions[frame_idx])
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(frame, f"Frame {frame_idx}", (x, max(20, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Salva imagem
    output_path = os.path.join(output_dir, f"frame_{frame_idx:04d}.png")
    cv2.imwrite(output_path, frame)

print(f"✅ {len(all_predictions)} frames salvos em: {output_dir}")


- Analisar segundo frame para descobrir step e região limite
- Melhorar a binarização (ver o vídeo e o notebook do igor)
- Descobrir o bug do tracking

In [ ]:
frame_path=image_paths[1]
frame_idx=1
frame = cv2.imread(frame_path)
if frame is None:
    print(f"❌ Frame {frame_idx} ({frame_path}) não carregado!")
    # continue

print(f"\n📷 Frame {frame_idx}: {frame_path}")

# --- PASSO 1: Gerar regiões de busca ---
search_gen = generate_search_regions_circular(
    prev_bbox=prev_bbox,
    frame_shape=frame.shape,
    search_region_scale=2,
    step_size=1
)

all_regions = list(search_gen)
print(f"🔍 Geradas {len(all_regions)} regiões para o frame {frame_idx}")

# --- PASSO 2: Classificar regiões ---
results = []
for i, region in enumerate(all_regions):
    patch_region = tracker_utils.extract_patch(frame, region)
    # if patch_region.size == 0:
    #     continue
    
    pattern_region = preprocess_frame(patch_region) 
    
    result = clus.classify([pattern_region.tolist()])[0]
    activation = result.get("activationDegree", -1)
    
    results.append({
        'region_id': i,
        'bbox': region,
        'activation': activation,
        'patch': pattern_region  # armazenamos para possível retraining
    })
    
if not results:
    print("⚠️ Nenhuma região válida encontrada, mantendo prev_bbox.")
    all_predictions.append(prev_bbox)
    # continue

# --- PASSO 3: Escolher melhor região ---
best_region = max(results, key=lambda r: r['activation'])
prev_bbox = best_region['bbox']
all_predictions.append(prev_bbox)

print(f"🎯 Melhor região frame {frame_idx}: {best_region['bbox']} ativação={best_region['activation']:.4f}")

# --- Visualizar patch da melhor região ---
best_patch = tracker_utils.extract_patch(frame, best_region['bbox'])

plt.figure()
plt.title(f"Frame {frame_idx} - Melhor Região")
plt.imshow(best_patch)